In [7]:
!pip install langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached pypdf-6.11.0-py3-none-any.whl.metadata (7.2 kB)
  Using cached pymupdf-1.27.2.3-cp310-abi3-win_amd64.whl.metadata (24 kB)
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 2.5/2.5 MB 13.8 MB/s  0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 12.7 MB/s  0:00:00
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   -------- ------------------------------- 4.2/19.2 MB 19.8 MB/s eta 0:00:01
   --------------- ------------------------ 7.6/19.2 MB 18.2 MB/s eta 0:00:01
   ----------------------- ---------------- 11.5/19.2 MB 18.2 MB/s eta 0:00:01
   ------------------------------ --------- 14.7/19.2 MB 18.0 MB/s eta 0:00:01
   ---------------------------------------  19.1/19.2 MB 18.2 MB/s eta 0:00:01
   -----------------------------------

In [106]:
from langchain_core.documents import Document

In [107]:
sample_doc = Document(
    page_content="hello wworld",
    metadata={"source":"https://www.google.com"}
)

In [108]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='hello wworld')

In [109]:
#text data
from langchain_community.document_loaders.text import TextLoader

loader=TextLoader("data/Python.txt",encoding="utf-8")

In [110]:
document=loader.load()

In [111]:
document

[Document(metadata={'source': 'data/Python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and mo

In [112]:
#pdf data
from langchain_community.document_loaders.pdf import PyPDFLoader

pdf_loader=PyPDFLoader("data/research.pdf")
document=pdf_loader.load()

In [113]:
document

[Document(metadata={'producer': 'pdfcpu v0.9.1 dev', 'creator': 'PyPDF', 'creationdate': '2026-03-21T04:11:43+00:00', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'book': 'Advances in Neural Information Processing Systems 30', 'created': '2017', 'date': '2017', 'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more paralleli

In [114]:
#ingestion pipeline 
#data => documents

import os
from langchain_community.document_loaders.pdf import PyPDFLoader
def load_all_pdfs():
    folder_path ="data/pdfs"
    num_docs=0
    all_docs=[]

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            pdf_path=os.path.join(folder_path,filename)
            loader=PyPDFLoader(pdf_path)
            doc=loader.load()
            all_docs.extend(doc)
            num_docs += 1
            
        print("total pdfs : ",num_docs)
    return all_docs

In [115]:
all_pdf_documents = load_all_pdfs()

total pdfs :  1


In [116]:
type(all_pdf_documents[0])

langchain_core.documents.base.Document

In [117]:
#chunks
!pip install langchain_text_splitters

In [118]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
def split_doc(documents,chunk_size=500,chunk_overlap=50):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunked_docs=text_splitter.split_documents(documents)
    return chunked_docs

In [119]:
chunks=split_doc(all_pdf_documents)

In [120]:
len(chunks)

244

In [121]:
#embedding
from sentence_transformers import SentenceTransformer


In [122]:
class EmbeddingManager:
    def __init__(self,model_name="all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=SentenceTransformer(self.model_name)
        print("embedding dimensions ",self.model.get_sentence_embedding_dimension())

    def generate_embedding(self,text):
        embeddings=self.model.encode(text,show_progress_bar=True)
        return embeddings

In [123]:
embedding_manager=EmbeddingManager()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions  384


C:\Users\rahul shakya\AppData\Local\Temp\ipykernel_1512\4137099929.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions ",self.model.get_sentence_embedding_dimension())


In [124]:
import chromadb
import uuid

In [125]:
#vector store first
class VectorStoreManager:
    def __ init__(self,
                 persist_directory="data/vector_store",
                 collection_name="pdf_documents"):

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):

        os.makedirs(self.persist_directory, exist_ok=True)

        # create a client
        self.client = chromadb.PersistentClient(
            path=self.persist_directory
        )

        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                "description": "vector store collection for pdf embeddings in RAG"
            }
        )
        print("initialized the vector store with collection : ",self.collection_name)
        print("docs in collection : ",self.collection.count())

    def add_documents(self,documents,embeddings):
        if len(documents) != len(embeddings):

            raise ValueError("num of documents does not match num of embeddings")
        #store => ids,embedding ,documents,metadata
        ids=[]
        all_metadata=[]
        documents_content=[]
        embeddings_list=[]

        for i , (doc,embeddings) in enumerate(zip(documents,embeddings)):
            doc_id=f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata=dict(doc.metadata)
            metadata["doc_index"]=i
            metadata["content_length"]=len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)
            embeddings_list.append(embeddings.tolist())
            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )
        print("total documents added in vector store",len(documents_content))
        print("docs in collection:",self.collection.count())

SyntaxError: expected '(' (1840165790.py, line 3)

In [ ]:
vector_store=VectorStoreManager()

In [ ]:
texts=[doc.page_content for doc in chunks]
embedding=embedding_manager.generate_embedding(texts)
vector_store.add_documents(chunks,embedding)

## Retrival

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5):
        query_embedding = self.embedding_manager.generate_embedding([query])[0]

        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )

        retrieved_docs = []

        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(
                zip(ids, metadatas, documents, distances)
            ):
                retrieved_docs.append({
                    "id": doc_id,
                    "document": document,
                    "metadata": metadata,
                    "distance": distance,
                    "rank": i + 1
                })

            print(f"retrieved {len(retrieved_docs)} documents")

        return retrieved_docs

In [ ]:
rag_retriever=RAGRetriever(embedding_manager,vector_store)

In [ ]:
rag_retriever.retrieve("what is rag")

# Integrate with LLMs

# GROQAI-GPT

In [ ]:
!pip install langchain-groq

In [ ]:
pip install python-dotenv

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("SigmaGPT/Backend/.env")

api_key = os.getenv("GROQ_API_KEY")

print(api_key)

In [ ]:
from langchain_groq import ChatGroq
import os

llm = ChatGroq(
    groq_api_key=api_key,
    model_name="llama-3.3-70b-versatile",
    temperature=0.1,
    max_tokens=1024
)

In [ ]:
#generate our retrival - augumented output
def generate_output(query, rag_retriever, llm, top_k=3):

    results = rag_retriever.retrieve(query, top_k)

    context = "\n".join(
        doc["document"] for doc in results
    ) if results else ""

    if not context:
        return "No relevant context found."

    # Context + Query Prompt
    prompt = f"""
    Use the given context to answer the question.

    Context:
    {context}

    Question:
    {query}
    """

    response = llm.invoke(prompt)

    return response.content

In [ ]:
answer=generate_output("what is encoder?",rag_retriever,llm)

In [ ]:
print(answer)